# Notebook 03 — Model Training
**Project:** IDS-Africa-ML  
**Author:** [Your Name]  
**Purpose:** Train Random Forest, XGBoost, and Neural Network models on the preprocessed CICIDS-2017 dataset and evaluate their performance.

---

### What This Notebook Does
```
Load preprocessed data from Drive
    → Train Random Forest
    → Train XGBoost
    → Train Neural Network
    → Evaluate all models (Accuracy, F1, Precision, Recall)
    → Plot Confusion Matrices
    → Plot ROC Curves
    → Build comparison table
    → Save all models to Drive
```

## Cell 1 — Mount Drive & Load Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, joblib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report,
                             confusion_matrix, roc_curve, auc)
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier
from tensorflow import keras
from tensorflow.keras import layers

import warnings
warnings.filterwarnings('ignore')

base = '/content/drive/MyDrive/IDS_Africa_ML/'
print("All libraries loaded")

## Cell 2 — Load Preprocessed Data from Drive

In [ ]:
X_train = np.load(base + 'outputs/X_train.npy')
X_test  = np.load(base + 'outputs/X_test.npy')
y_train = np.load(base + 'outputs/y_train.npy')
y_test  = np.load(base + 'outputs/y_test.npy')
le      = joblib.load(base + 'models/label_encoder.pkl')

print("DATA LOADED FROM DRIVE")
print("=" * 40)
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")
print(f"\nClasses : {list(le.classes_)}")
print(f"\nReady for model training")

## Cell 3 — Helper Function: Evaluate Any Model

In [ ]:
def evaluate_model(name, y_true, y_pred, train_time, model_size_mb):
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"\nRESULTS — {name}")
    print("=" * 50)
    print(f"  Accuracy          : {acc*100:.2f}%")
    print(f"  F1 Score          : {f1*100:.2f}%")
    print(f"  Precision         : {prec*100:.2f}%")
    print(f"  Recall            : {rec*100:.2f}%")
    print(f"  Training Time     : {train_time:.2f} seconds")
    print(f"  Model Size        : {model_size_mb:.2f} MB")
    print(f"\nPer-Class Report:")
    print(classification_report(y_true, y_pred,
                                target_names=le.classes_,
                                zero_division=0))
    return {
        "Model"         : name,
        "Accuracy"      : round(acc * 100, 2),
        "F1 Score"      : round(f1  * 100, 2),
        "Precision"     : round(prec * 100, 2),
        "Recall"        : round(rec  * 100, 2),
        "Train Time (s)": round(train_time, 2),
        "Size (MB)"     : round(model_size_mb, 2),
    }

print("Evaluation function ready")

## Cell 4 — Train Model 1: Random Forest

In [ ]:
print("Training Random Forest...")
print("(This may take 3-5 minutes)")
print("-" * 40)

start = time.time()
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
rf.fit(X_train, y_train)
rf_time = time.time() - start

# Predict
rf_pred = rf.predict(X_test)

# Save model
rf_path = base + 'models/random_forest.pkl'
joblib.dump(rf, rf_path)
rf_size = os.path.getsize(rf_path) / (1024 * 1024)

# Evaluate
rf_results = evaluate_model(
    "Random Forest", y_test, rf_pred, rf_time, rf_size
)

print(f"\nRandom Forest saved ({rf_size:.2f} MB)")

## Cell 5 — Train Model 2: XGBoost

In [ ]:
print("Training XGBoost...")
print("(This may take 2-4 minutes)")
print("-" * 40)

# Map labels to 0-based consecutive integers (XGBoost requirement)
from sklearn.preprocessing import LabelEncoder
le_xgb = LabelEncoder()
y_train_xgb = le_xgb.fit_transform(y_train)
y_test_xgb  = le_xgb.transform(y_test)

start = time.time()
xgb = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric='mlogloss',
    use_label_encoder=False,
    n_jobs=-1
)
xgb.fit(X_train, y_train_xgb)
xgb_time = time.time() - start

# Predict (map back to original labels)
xgb_pred_encoded = xgb.predict(X_test)
xgb_pred = le_xgb.inverse_transform(xgb_pred_encoded)

# Save model
xgb_path = base + 'models/xgboost.pkl'
joblib.dump(xgb, xgb_path)
xgb_size = os.path.getsize(xgb_path) / (1024 * 1024)

# Evaluate
xgb_results = evaluate_model(
    "XGBoost", y_test, xgb_pred, xgb_time, xgb_size
)

print(f"\nXGBoost saved ({xgb_size:.2f} MB)")

## Cell 6 — Train Model 3: Neural Network

In [ ]:
print("Training Neural Network...")
print("(This may take 3-5 minutes)")
print("-" * 40)

n_classes = len(np.unique(y_train))
n_features = X_train.shape[1]

# Build model
nn = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(n_classes, activation='softmax')
])

nn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

nn.summary()

# Remap labels for Neural Network (needs 0-based consecutive)
from sklearn.preprocessing import LabelEncoder
le_nn = LabelEncoder()
y_train_nn = le_nn.fit_transform(y_train)
y_test_nn  = le_nn.transform(y_test)

start = time.time()
history = nn.fit(
    X_train, y_train_nn,
    epochs=20,
    batch_size=512,
    validation_split=0.1,
    verbose=1
)
nn_time = time.time() - start

# Predict
nn_pred_proba = nn.predict(X_test)
nn_pred_encoded = np.argmax(nn_pred_proba, axis=1)
nn_pred = le_nn.inverse_transform(nn_pred_encoded)

# Save model
nn_path = base + 'models/neural_network.h5'
nn.save(nn_path)
nn_size = os.path.getsize(nn_path) / (1024 * 1024)

# Evaluate
nn_results = evaluate_model(
    "Neural Network", y_test, nn_pred, nn_time, nn_size
)

print(f"\nNeural Network saved ({nn_size:.2f} MB)")

## Cell 7 — Plot Training History (Neural Network)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train', color='#2ecc71', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val',   color='#e74c3c', linewidth=2)
axes[0].set_title('Neural Network — Accuracy per Epoch', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train', color='#2ecc71', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val',   color='#e74c3c', linewidth=2)
axes[1].set_title('Neural Network — Loss per Epoch', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(base + 'figures/03a_nn_training_history.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved")

## Cell 8 — Model Comparison Table

In [ ]:
results_df = pd.DataFrame([rf_results, xgb_results, nn_results])
results_df = results_df.set_index('Model')

print("MODEL COMPARISON TABLE")
print("=" * 70)
print(results_df.to_string())

# Save to CSV — this becomes Table 1 in your paper
results_df.to_csv(base + 'outputs/model_comparison.csv')

# Plot comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

metrics   = ['Accuracy', 'F1 Score', 'Precision']
colors    = ['#3498db', '#e74c3c', '#2ecc71']
models    = results_df.index.tolist()

for i, metric in enumerate(metrics):
    vals = results_df[metric].values
    bars = axes[i].bar(models, vals, color=colors, edgecolor='white', linewidth=1.5)
    axes[i].set_title(metric, fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Score (%)')
    axes[i].set_ylim(0, 110)
    axes[i].grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 1,
                     f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Model Performance Comparison — CICIDS-2017',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(base + 'figures/03b_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("\nComparison table saved")
print("Figure saved")

## Cell 9 — Confusion Matrices (All Three Models)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

models_info = [
    ("Random Forest", rf_pred),
    ("XGBoost",       xgb_pred),
    ("Neural Network",nn_pred),
]

class_names = le.classes_

for ax, (name, pred) in zip(axes, models_info):
    cm = confusion_matrix(y_test, pred)
    
    # Normalize for readability
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    sns.heatmap(cm_norm, annot=True, fmt='.2f',
                xticklabels=class_names,
                yticklabels=class_names,
                cmap='Blues', ax=ax,
                linewidths=0.5, linecolor='white')
    ax.set_title(f'{name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('True Label', fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.suptitle('Confusion Matrices — Normalized (Row = True Class)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(base + 'figures/03c_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved")

## Cell 10 — ROC Curves (All Three Models)

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from itertools import cycle

unique_classes = np.unique(y_test)
n_cls = len(unique_classes)

# Binarize y_test
y_test_bin = label_binarize(y_test, classes=unique_classes)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

models_roc = [
    ("Random Forest",  rf.predict_proba(X_test)),
    ("XGBoost",        xgb.predict_proba(X_test)),
    ("Neural Network", nn.predict(X_test)),
]

colors = cycle(['#e74c3c','#3498db','#2ecc71','#9b59b6',
                '#f39c12','#1abc9c','#e67e22','#34495e','#e91e63'])

for ax, (name, y_score) in zip(axes, models_roc):
    # Handle XGBoost label mapping
    if name == "XGBoost":
        y_score_full = np.zeros((len(y_test), n_cls))
        for j, cls in enumerate(unique_classes):
            idx = np.where(le_xgb.classes_ == cls)[0]
            if len(idx) > 0:
                y_score_full[:, j] = y_score[:, idx[0]]
        y_score = y_score_full

    # Handle Neural Network label mapping
    if name == "Neural Network":
        y_score_full = np.zeros((len(y_test), n_cls))
        for j, cls in enumerate(unique_classes):
            idx = np.where(le_nn.classes_ == cls)[0]
            if len(idx) > 0:
                y_score_full[:, j] = y_score[:, idx[0]]
        y_score = y_score_full

    macro_auc_list = []
    for i, color in zip(range(n_cls), colors):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        macro_auc_list.append(roc_auc)
        ax.plot(fpr, tpr, color=color, lw=1.2,
                label=f'{le.classes_[unique_classes[i]]} (AUC={roc_auc:.2f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=10)
    ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(f'{name}\nMacro AUC = {np.mean(macro_auc_list):.3f}',
                 fontsize=11, fontweight='bold')
    ax.legend(loc='lower right', fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(base + 'figures/03d_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved")

## Cell 11 — Save Final Summary

In [ ]:
# Final summary printout
print("=" * 60)
print("NOTEBOOK 03 COMPLETE — FINAL SUMMARY")
print("=" * 60)

print("\nMODEL COMPARISON:")
print(results_df[['Accuracy','F1 Score','Train Time (s)','Size (MB)']].to_string())

print("\nFILES SAVED TO DRIVE:")
saved = [
    ('models/', 'random_forest.pkl'),
    ('models/', 'xgboost.pkl'),
    ('models/', 'neural_network.h5'),
    ('outputs/', 'model_comparison.csv'),
    ('figures/', '03a_nn_training_history.png'),
    ('figures/', '03b_model_comparison.png'),
    ('figures/', '03c_confusion_matrices.png'),
    ('figures/', '03d_roc_curves.png'),
]
for folder, fname in saved:
    path = base + folder + fname
    exists = os.path.exists(path)
    size = os.path.getsize(path)/1024 if exists else 0
    print(f"  {'OK' if exists else 'MISSING':7s} {folder}{fname} ({size:.1f} KB)")

print("\n" + "=" * 60)
print("Next step: Notebook 04 — SHAP Explainability Analysis")
print("=" * 60)